# Module 6 lab: a first quantum algorithm, Bernstein-Vazirani

**Quantum Computing Foundations · Module 6 · about 75 minutes**

A box hides a string of bits s. You may ask it questions: give it a string x, and it answers one bit, s·x mod 2. A classical computer needs one question per bit of s. The Bernstein-Vazirani algorithm finds all of s with **one** question. In this lab you write three functions:

1. `classical_bv(query, n)`: the classical method, one query per bit;
2. `bv_oracle(s)`: the quantum version of the box, as a circuit;
3. `bv_circuit(oracle, n)`: the Bernstein-Vazirani circuit, which finds the hidden string of any oracle with one use of it.

Then you run the same circuit as the Deutsch-Jozsa algorithm, compare the number of queries, and see what noise does to it. The **Module 6 lab check** asks for results from this notebook, so keep it open in a second tab. Run each code cell with **Shift + Enter**, in order.

## Step 0: start Python

Run the cell below. The first time, Python can take up to a minute to start in your browser. qsim 1.4.0 adds `QuantumCircuit.compose`, which joins two circuits as in Qiskit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt   # used for the plots below
import qsim
from qsim import (QuantumCircuit, Statevector, Operator, AerSimulator, plot_histogram, bloch_vectors,
                  NoiseModel, depolarizing_error)

sim = AerSimulator(seed_simulator=7)


def f_hidden(s, x):
    """Prepared for you: s . x mod 2, the parity of the positions where both strings have a 1."""
    return sum(int(a) * int(b) for a, b in zip(s, x)) % 2


print("Ready. qsim", qsim.__version__, "and NumPy", np.__version__)

## Step 1: the problem, and the classical answer

Strings are written like Qiskit results: the **rightmost** character belongs to qubit 0. The hidden function is

f(x) = s·x mod 2 = (s<sub>n−1</sub>x<sub>n−1</sub> + … + s<sub>1</sub>x<sub>1</sub> + s<sub>0</sub>x<sub>0</sub>) mod 2.

Run the first cell: it prints f(x) for s = 1011 and all 16 inputs. Canvas asks how many of them give f(x) = 1.

In [ ]:
s = "1011"
print("x     f(x)")
for k in range(16):
    x = format(k, "04b")
    print(x, "   ", f_hidden(s, x))
ones = sum(f_hidden(s, format(k, "04b")) for k in range(16))
print(f"\n{ones} of the 16 inputs give f(x) = 1")

A classical computer can only ask f about one x at a time, and each answer is one bit. The best strategy asks about the inputs with a single 1: the input with a 1 in position j answers with s[j]. That takes **n queries** for an n-bit string, and no classical method can do it with fewer, because each answer carries one bit of information.

Complete `classical_bv(query, n)`. `query` is a function: `query("0100")` returns f for x = 0100. Return the hidden string as a text string, such as `"1011"`. Then run the cell: it tests your function on a hidden 8-bit string and counts the queries.

In [ ]:
def classical_bv(query, n):
    # YOUR CODE: replace the next line. Ask n questions, one per bit, and return s as a string.
    raise NotImplementedError("Complete classical_bv() first.")


def make_query(secret):
    """A classical box for a hidden string, which counts how often it is asked."""
    def query(x):
        query.calls += 1
        return f_hidden(secret, x)
    query.calls = 0
    return query


box = make_query("10110010")
found = classical_bv(box, 8)
print("hidden string found:", found, "  queries used:", box.calls)

## Step 2: phase kickback

The quantum version of the box is a circuit called an **oracle**. It acts on n input qubits and one extra **answer qubit** (qubit n): |x⟩|y⟩ → |x⟩|y ⊕ f(x)⟩. On its own that just writes f(x) into the answer qubit. The trick is to put the answer qubit in |−⟩ first. Then flipping it does nothing to it, but multiplies the state by −1: the answer goes into the **phase** of the input, as (−1)<sup>f(x)</sup>|x⟩. This is **phase kickback**.

Run the cell. It puts qubit 0 in |+⟩ and the answer qubit 1 in |−⟩, applies one CNOT, and prints both Bloch vectors before and after.

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)              # input qubit 0 in |+>
qc.x(1)
qc.h(1)              # answer qubit 1 in |->
before = bloch_vectors(Statevector(qc))
qc.cx(0, 1)
after = bloch_vectors(Statevector(qc))
for q, name in ((0, "input qubit 0 "), (1, "answer qubit 1")):
    print(f"{name}: before {np.round(before[q], 3) + 0}   after {np.round(after[q], 3) + 0}")

The CNOT targets the answer qubit, yet the answer qubit is unchanged and the **control** turned from |+⟩ (Bloch vector along +x) into |−⟩ (along −x). The −1 for the |1⟩ part of the input was kicked back from the target to the control.

## Step 3: the oracle as a circuit

For f(x) = s·x mod 2, the oracle is simple: for every 1 in s, a CNOT from that input qubit to the answer qubit. Each CNOT adds x<sub>i</sub> to the answer, so together they add s·x mod 2. Remember the order: character `s[n - 1 - i]` belongs to qubit i.

Complete `bv_oracle(s)`. It returns a circuit with n + 1 qubits (inputs 0 to n − 1, answer qubit n) and no measurements. The cell checks it against f for all 16 inputs of s = 1011 and draws it.

In [ ]:
def bv_oracle(s):
    # YOUR CODE: replace the next line. Return a QuantumCircuit with len(s) + 1 qubits.
    raise NotImplementedError("Complete bv_oracle() first.")


s = "1011"
n = len(s)
oracle = bv_oracle(s)
right = 0
for k in range(2 ** n):
    x = format(k, f"0{n}b")
    qc = QuantumCircuit(n + 1)
    for i in range(n):
        if x[n - 1 - i] == "1":
            qc.x(i)                      # prepare the input |x>, answer qubit |0>
    out = Statevector(qc.compose(oracle)).probabilities_dict()
    result = list(out)[0]                # one basis state; its leftmost bit is the answer qubit
    right += int(result[0]) == f_hidden(s, x)
print(f"the oracle wrote f(x) correctly for {right} of {2 ** n} inputs")
print(oracle.draw())

## Step 4: the Bernstein-Vazirani circuit

The algorithm:

1. Put the answer qubit in |−⟩ (X, then H) and every input qubit in |+⟩ (H).
2. Use the oracle **once**. By phase kickback the input register becomes the sum of (−1)<sup>s·x</sup>|x⟩ over all x.
3. Put H on every input qubit again. The signs interfere so that everything cancels except |s⟩.
4. Measure the n input qubits: qubit i into classical bit i.

Complete `bv_circuit(oracle, n)`. It receives an oracle whose hidden string it does **not** know, adds it once with `qc.compose(oracle)`, and returns a circuit with n + 1 qubits and n classical bits. Then run the cell. It first shows the signs after step 2 for s = 101, then finds 1011, then a mystery 8-bit string. Canvas asks for the mystery string as a decimal number.

In [ ]:
def bv_circuit(oracle, n):
    # YOUR CODE: replace the next line. Return a QuantumCircuit(n + 1, n) that uses the oracle once.
    raise NotImplementedError("Complete bv_circuit() first.")


# The signs after the oracle, for s = 101: the input register is the sum of (-1)^(s.x) |x> / sqrt(8)
s = "101"
prep = QuantumCircuit(4)
prep.x(3)
prep.h(range(4))
amps = Statevector(prep.compose(bv_oracle(s))).data
signs = np.sign(np.real(amps[:8]))     # answer qubit |-> = (|0> - |1>)/sqrt(2): its |0> half
for k in range(8):
    x = format(k, "03b")
    print(f"x = {x}: sign {'+' if signs[k] > 0 else '-'}   (s.x mod 2 = {f_hidden(s, x)})")

counts = sim.run(bv_circuit(bv_oracle("1011"), 4), shots=1000).result().get_counts()
print("\nhidden string 1011:", counts)


def mystery_oracle():
    """Prepared for you: an oracle for a hidden 8-bit string."""
    return bv_oracle(format(181, "08b"))


counts = sim.run(bv_circuit(mystery_oracle(), 8), shots=1000).result().get_counts()
found = max(counts, key=counts.get)
print("mystery oracle:", counts, "  as a decimal number:", int(found, 2))

One query found all of s. Each query to the quantum oracle acts on a superposition of all 2<sup>n</sup> inputs at once, and the final H gates turn the pattern of signs into the answer. Note that the circuit never "reads" all the values f(x): only the interference pattern matters.

## Step 5 (guided): Deutsch-Jozsa, the same circuit

The **Deutsch-Jozsa** problem: f is promised to be either **constant** (the same answer for every x) or **balanced** (0 for exactly half of the inputs). Which is it? Classically, in the worst case you must check more than half of the inputs. The Deutsch-Jozsa algorithm uses exactly the Bernstein-Vazirani circuit: if f is constant, every shot gives 00…0; if it is balanced, no shot does.

Run the cell. It tries five prepared 3-bit oracles with your `bv_circuit()`: A and E are constant, B, C and D are balanced. D is not of the form s·x, so its results are spread out, but they still never include 000.

In [ ]:
def dj_oracle(kind, n=3):
    """Prepared for you: five oracles for f on 3 bits, with the answer qubit 3."""
    qc = QuantumCircuit(n + 1)
    if kind == "A":
        qc.x(n)                        # f(x) = 1 for every x: constant
    elif kind == "B":
        qc.cx(1, n)                    # f(x) = x1: balanced
    elif kind == "C":
        qc.cx(0, n)
        qc.cx(2, n)                    # f(x) = x0 XOR x2: balanced
    elif kind == "D":
        qc.ccx(0, 1, n)
        qc.cx(2, n)                    # f(x) = (x0 AND x1) XOR x2: balanced, not of the form s.x
    return qc                          # "E": f(x) = 0 for every x: constant


for kind in "ABCDE":
    ones = 0
    for k in range(8):                 # the classical truth table, to label the oracle
        x = format(k, "03b")
        qc = QuantumCircuit(4)
        for i in range(3):
            if x[2 - i] == "1":
                qc.x(i)
        ones += int(list(Statevector(qc.compose(dj_oracle(kind))).probabilities_dict())[0][0])
    label = "constant" if ones in (0, 8) else "balanced" if ones == 4 else "neither"
    c = sim.run(bv_circuit(dj_oracle(kind), 3), shots=1000).result().get_counts()
    print(f"oracle {kind}: f(x) = 1 for {ones} of 8 inputs ({label:8s})  P(000) = {c.get('000', 0) / 1000:.3f}   {c}")

## Step 6: counting queries

The table below compares the number of queries for n-bit inputs. For Bernstein-Vazirani, a classical computer needs n queries and the quantum circuit 1: a large saving, but it grows only in proportion to n. For Deutsch-Jozsa, a classical computer that must be **certain** needs 2<sup>n−1</sup> + 1 queries in the worst case, against 1: an exponential gap. (A classical computer that may be wrong with a small probability needs only a few random queries, which is why Deutsch-Jozsa is a demonstration rather than a practical speedup.)

In [ ]:
ns = np.arange(1, 11)
print(" n   BV classical   DJ classical (certain, worst case)   quantum")
for n in ns:
    print(f"{n:2d}   {n:12d}   {2 ** (n - 1) + 1:34d}   {1:7d}")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.semilogy(ns, ns, "o-", color="#24313D", label="Bernstein-Vazirani, classical")
ax.semilogy(ns, 2 ** (ns - 1) + 1, "s-", color="#6A626B", label="Deutsch-Jozsa, classical and certain")
ax.semilogy(ns, np.ones_like(ns), "D-", color="#99004C", label="quantum, both problems")
ax.set_xlabel("number of input bits n")
ax.set_ylabel("queries")
ax.legend(fontsize=8)
plt.show()

## Step 7: the algorithm with noise

Real hardware adds errors (Module 5). Run the circuit for s = 111111 (six CNOTs in the oracle) with a depolarizing error λ on every CNOT, 1,000 shots and seed 7, and count how often it still returns the hidden string. Canvas asks for the count at λ = 0.05.

In [ ]:
s = "111111"
for lam in (0, 0.01, 0.02, 0.05, 0.10):
    model = NoiseModel()
    model.add_all_qubit_quantum_error(depolarizing_error(lam, 2), ["cx"])
    noisy = AerSimulator(seed_simulator=7, noise_model=model)
    c = noisy.run(bv_circuit(bv_oracle(s), 6), shots=1000).result().get_counts()
    print(f"lambda = {lam:.2f}: hidden string returned in {c.get(s, 0):4d} of 1000 shots")

Each CNOT in the oracle is one more chance for an error, so a hidden string with more 1s is found less reliably. The most common result is still the right one at these error rates, so repeating the run and taking the most frequent answer works; for larger circuits the errors add up quickly, which is why error rates matter so much.

## Step 8: your personal oracle

Open the Canvas quiz **Module 6 lab check**. Question 1 gives you two numbers: **KEY** (which chooses a hidden 6-bit string and the random seed) and **DEP_PCT** (a depolarizing error on each CNOT, in percent). Type them below in place of `None` and run the cell.

In [ ]:
KEY = None       # for example: KEY = 512
DEP_PCT = None   # for example: DEP_PCT = 5

if None in (KEY, DEP_PCT):
    print("Enter KEY and DEP_PCT from Canvas first.")
else:
    print(f"KEY = {KEY} chooses your hidden 6-bit string; the check cell runs the Bernstein-Vazirani circuit "
          f"for it with a {DEP_PCT}% depolarizing error on each CNOT.")

## Step 9: run the check cell

**Do not edit this cell.** Run it. It tests your three functions on cases you have not seen, including hidden oracles. If they all pass, it runs the Bernstein-Vazirani circuit for your personal oracle 1,000 times with the noise from Step 8 and prints your **verification value**: the number of shots that returned your hidden string. Type it into Canvas question 1.

If a test does not pass, read its message, fix that function, run its cell again, then run this cell again.

In [ ]:
# CHECK CELL: do not edit
import numpy as np
from qsim import AerSimulator, NoiseModel, Operator, QuantumCircuit, depolarizing_error

N_PERSONAL = 6


def reference_oracle(s):
    """|x>|y> -> |x>|y XOR s.x>: a CNOT from input qubit i to the answer qubit n for every 1 in s."""
    n = len(s)
    qc = QuantumCircuit(n + 1)
    for i in range(n):
        if s[n - 1 - i] == "1":
            qc.cx(i, n)
    return qc


def reference_circuit(oracle, n):
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(range(n + 1))
    qc = qc.compose(oracle)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc


def personal_string(key):
    return format((int(key) * 47) % 63 + 1, f"0{N_PERSONAL}b")


def personal_value(key, dep_pct):
    s = personal_string(key)
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(float(dep_pct) / 100, 2), ["cx"])
    qc = reference_circuit(reference_oracle(s), N_PERSONAL)
    counts = AerSimulator(seed_simulator=int(key), noise_model=nm).run(qc, shots=1000).result().get_counts()
    return int(counts.get(s, 0))


def _marked_oracle(s, variant):
    """A hidden oracle for testing bv_circuit(): it starts with an 'id' marker on the answer qubit."""
    n = len(s)
    qc = QuantumCircuit(n + 1)
    qc.id(n)
    ones = [i for i in range(n) if s[n - 1 - i] == "1"]
    if variant == "cz" and ones:                 # the same oracle written with CZ gates
        qc.h(n)
        for i in ones:
            qc.cz(i, n)
        qc.h(n)
    else:
        for i in (reversed(ones) if variant == "rev" else ones):
            qc.cx(i, n)
    return qc


def check_module6(bv_oracle, bv_circuit, classical_bv, KEY, DEP_PCT):
    msgs = []

    # Test 1: bv_oracle(s) is the oracle |x>|y> -> |x>|y XOR s.x>, with the answer qubit n.
    for s in ["1", "0", "10", "01", "101", "1101", "0110", "11111", "10010", "000"]:
        n = len(s)
        try:
            qc = bv_oracle(s)
        except Exception as exc:  # noqa: BLE001
            return False, [f"bv_oracle('{s}') stopped with an error: {exc}"], None
        if not isinstance(qc, QuantumCircuit) or qc.num_qubits != n + 1:
            return False, [f"bv_oracle('{s}') must return a QuantumCircuit with {n + 1} qubits: "
                           f"{n} input qubits (0 to {n - 1}) and the answer qubit {n}."], None
        if any(i.name in ("measure", "reset") for i in qc.data):
            return False, [f"bv_oracle('{s}') must not measure or reset: it is only the oracle."], None
        if not Operator(qc).equiv(Operator(reference_oracle(s))):
            hint = ("For every '1' in s, add a CNOT from that input qubit to the answer qubit n. The rightmost "
                    "character of s belongs to qubit 0, so character s[n - 1 - i] belongs to qubit i.")
            if Operator(qc).equiv(Operator(reference_oracle(s[::-1]))):
                hint = "The bits are in the wrong order. " + hint
            return False, [f"bv_oracle('{s}') is not the right oracle. {hint}"], None
    msgs.append("Test 1, bv_oracle(): passed for 10 hidden strings")

    # Test 2: bv_circuit(oracle, n) finds a hidden string it is not told, with one use of the oracle.
    rng = np.random.default_rng(606)
    cases = [("1011", "cx"), ("0000", "cx"), ("111", "cz"), ("100101", "rev")]
    for n in (2, 3, 5, 6):
        cases.append(("".join(str(b) for b in rng.integers(0, 2, size=n)), "cx"))
    for s, variant in cases:
        n = len(s)
        oracle = _marked_oracle(s, variant)
        try:
            qc = bv_circuit(oracle.copy(), n)
        except Exception as exc:  # noqa: BLE001
            return False, [f"bv_circuit(oracle, {n}) stopped with an error: {exc}"], None
        if not isinstance(qc, QuantumCircuit) or qc.num_qubits != n + 1:
            return False, [f"bv_circuit(oracle, {n}) must return a QuantumCircuit with {n + 1} qubits."], None
        uses = sum(1 for i in qc.data if i.name == "id")
        if uses != 1:
            return False, [f"bv_circuit() used the oracle {uses} times. Bernstein-Vazirani needs it exactly once: "
                           "add it with qc.compose(oracle), and do not build your own oracle inside bv_circuit()."], None
        if qc.num_clbits != n or sum(1 for i in qc.data if i.name == "measure") != n:
            return False, [f"bv_circuit(oracle, {n}) must have {n} classical bits and measure the {n} input qubits "
                           f"(0 to {n - 1}) into them, and not the answer qubit."], None
        try:
            counts = AerSimulator(seed_simulator=1).run(qc, shots=200).result().get_counts()
        except Exception as exc:  # noqa: BLE001
            return False, [f"Running bv_circuit(oracle, {n}) stopped with an error: {exc}"], None
        if counts != {s: 200}:
            hint = ("Prepare the answer qubit in |-> (X, then H), put H on every input qubit, add the oracle, put H "
                    "on the input qubits again, then measure input qubit i into classical bit i.")
            if counts == {s[::-1]: 200}:
                hint = "The result is reversed: measure input qubit i into classical bit i. " + hint
            elif len(counts) > 1:
                hint = "The result is random, so the interference is missing. " + hint
            return False, [f"For a hidden oracle, bv_circuit() gave {dict(sorted(counts.items())[:4])}, but every shot "
                           f"should give the hidden string. {hint}"], None
    msgs.append("Test 2, bv_circuit(): passed, finding 8 hidden strings with one use of the oracle each")

    # Test 3: classical_bv(query, n) finds s with n queries.
    for s in ["1", "01", "110", "1011", "100110", "01111110"]:
        n = len(s)
        calls = []

        def query(x, s=s, n=n, calls=calls):
            x = str(x)
            if len(x) != n or set(x) - set("01"):
                raise ValueError(f"query() needs a string of {n} characters, each 0 or 1, such as {'0' * (n - 1) + '1'!r}")
            calls.append(x)
            return sum(int(a) * int(b) for a, b in zip(s, x)) % 2

        try:
            got = classical_bv(query, n)
        except Exception as exc:  # noqa: BLE001
            return False, [f"classical_bv(query, {n}) stopped with an error: {exc}"], None
        if str(got) != s:
            hint = ("Ask about one bit at a time: the input with a single 1 in position j returns s[j]. Join the "
                    "answers into a string in the same order.")
            if str(got) == s[::-1]:
                hint = "The answer is reversed. " + hint
            return False, [f"classical_bv(query, {n}) returned {got!r}, but the hidden string is {s!r}. {hint}"], None
        if len(calls) > n:
            return False, [f"classical_bv(query, {n}) used {len(calls)} queries. Use exactly {n}: one per bit."], None
    msgs.append("Test 3, classical_bv(): passed, each with n queries")

    if KEY is None or DEP_PCT is None:
        return False, msgs + ["Enter KEY and DEP_PCT from Canvas in Step 8, then run this cell again."], None
    value = personal_value(KEY, DEP_PCT)
    msgs.append(f"Your personal oracle has a 6-bit hidden string. The Bernstein-Vazirani circuit for it was run "
                f"1,000 times with seed {KEY} and a {DEP_PCT}% depolarizing error on each CNOT.")
    return True, msgs, value


ok, messages, value = check_module6(bv_oracle, bv_circuit, classical_bv, KEY, DEP_PCT)
for m in messages:
    print(m)
if ok:
    print(f"\nAll tests passed. Your verification value for KEY = {KEY}, DEP_PCT = {DEP_PCT}: {value}")
    print("Type this number into Canvas question 1.")
else:
    print("\nNot passed yet: no verification value.")


## Finish

1. Answer the five questions of the **Module 6 lab check** in Canvas and submit.
2. Take the **Module 6 quiz**.
3. Fill in the short **Module 6 time log**.

Your notebook is saved in this browser as you work. It is not saved anywhere else, so to keep a copy choose **File > Download**. Next is Course Completion: the final quiz and the final coding task, which extends your Bell circuit to three qubits (a GHZ state).